<a href="https://colab.research.google.com/github/ShreyaMathur19/Clustered-Diagonal-Segment-Format-CDSF-/blob/main/CDS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**CDS (Compressed Diagonal Storage)**

CDS stores matrix elements along diagonals using offsets and a dense diagonal representation. We converted the matrix from COO to CDS, performed SpMV, and measured execution time and peak memory usage to evaluate efficiency for banded matrices.

In [4]:
from scipy.io import mmread
import numpy as np
from scipy.sparse import coo_matrix
from time import perf_counter
import numba as nb
from memory_profiler import memory_usage

In [3]:
!pip install memory-profiler

In [ ]:
# Load the matrix from an .mtx file
A = mmread("wang3.mtx")
A = coo_matrix(A)

In [ ]:
#COO to CDS
def encode_cds(A):
    """
    Encode sparse matrix A (COO/any scipy sparse) into CDS format.

    CDS format:
      - offsets[k] = diagonal offset d = col - row
      - vals[k, i] = A[i, i + d] if valid, else 0

    Returns:
        offsets : int32 array of shape (num_diagonals,)
        vals    : float64 array of shape (num_diagonals, nrows)

    Notes:
        - Diagonals are stored in ascending offset order
        - Works for square and rectangular matrices
    """
    A = A.tocoo(copy=False)
    if hasattr(A, "sum_duplicates"):
        A.sum_duplicates()

    nrows, ncols = A.shape

    row = A.row.astype(np.int32, copy=False)
    col = A.col.astype(np.int32, copy=False)
    data = A.data.astype(np.float64, copy=False)

    nnz = data.size
    if nnz == 0:
        return np.empty(0, dtype=np.int32), np.zeros((0, nrows), dtype=np.float64)

    # diagonal offsets
    offsets = np.unique((col - row).astype(np.int32))
    offsets.sort()

    # map offset -> row index in vals
    offset_to_idx = {d: k for k, d in enumerate(offsets)}

    # CDS storage: one row per diagonal, one column per matrix row
    vals = np.zeros((len(offsets), nrows), dtype=np.float64)

    # fill
    for r, c, v in zip(row, col, data):
        d = c - r
        k = offset_to_idx[d]
        vals[k, r] = v

    return offsets.astype(np.int32), vals

In [ ]:
#CDSF SPMV

@nb.njit(cache=True, fastmath=True)
def cds_spmv_numba(offsets, vals, nrows, ncols, x):
    """
    Compute y = A @ x using CDS storage.

    Parameters:
        offsets : int32 [num_diagonals]
        vals    : float64 [num_diagonals, nrows]
        nrows, ncols : matrix shape
        x       : float64 [ncols]

    Returns:
        y       : float64 [nrows]
    """
    y = np.zeros(nrows, dtype=np.float64)
    ndiags = offsets.shape[0]

    for k in range(ndiags):
        d = offsets[k]

        # valid row range so that j = i + d is inside [0, ncols)
        i_start = 0
        if d < 0:
            i_start = -d

        i_end = nrows
        if ncols - d < i_end:
            i_end = ncols - d

        for i in range(i_start, i_end):
            y[i] += vals[k, i] * x[i + d]

    return y

In [ ]:
#SPMV time

def time_cds_spmv_numba(offsets, vals, shape, reps=20, seed=42):
    """
    Times y = A @ x for CDS (Numba kernel).

    Inputs:
      - offsets: int32 array
      - vals: float64 2D array [num_diagonals, nrows]
      - shape: (nrows, ncols)

    Returns:
      ms per call (float)
    """
    nrows, ncols = shape
    rng = np.random.default_rng(seed)
    x = rng.standard_normal(ncols).astype(np.float64)

    # Warm-up #1: JIT compile
    _ = cds_spmv_numba(offsets, vals, nrows, ncols, x)

    # Warm-up #2: steady-state run
    _ = cds_spmv_numba(offsets, vals, nrows, ncols, x)

    # Timed runs
    t0 = perf_counter()
    for _ in range(reps):
        y = cds_spmv_numba(offsets, vals, nrows, ncols, x)
    t_ms = (perf_counter() - t0) / reps * 1000.0

    print(f"CDS SpMV (Numba): {t_ms:.2f} ms per call  |  shape={shape}, diagonals={len(offsets)}, vals_shape={vals.shape}")
    return t_ms

In [2]:
#Time and peak memory for COO to CDS
def time_and_peak_mem_coo_to_cds(A_coo, reps=5, interval=0.0001):
    """
    Measures average time (ms) and peak memory (KB) for COO->CDS encoding.
    Uses memory_profiler to capture peak RSS delta.
    """
    times_ms = []
    peak_deltas_kb = []
    arrays = None

    # Warm-up
    _ = encode_cds(A_coo)

    for run in range(reps):
        t0 = perf_counter()
        mem_trace, arrays = memory_usage(
            (encode_cds, (A_coo,), {}),
            retval=True,
            interval=interval
        )
        elapsed_ms = (perf_counter() - t0) * 1000.0
        peak_delta_kb = (max(mem_trace) - mem_trace[0]) * 1024

        print(
            f"Run {run+1}: start={mem_trace[0]*1024:.2f} MB, "
            f"peak={max(mem_trace)*1024:.2f} MB, "
            f"delta={peak_delta_kb:.2f} KB, "
            f"time={elapsed_ms:.2f} ms"
        )

        times_ms.append(elapsed_ms)
        peak_deltas_kb.append(peak_delta_kb)

    avg_ms = float(np.mean(times_ms))
    peak_kb = float(max(peak_deltas_kb))

    print("\n=== Summary COO->CDS ===")
    print(f"Average time: {avg_ms:.2f} ms per call (over {reps})")
    print(f"Peak memory during conversion: {peak_kb:.2f} KB")

    return arrays, avg_ms, peak_kb